In [164]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, date
from pandas.core.config_init import register_converter_doc
import zipfile

In [165]:
def get_html(url):
    html = None
    try:
        # Send a GET request to the URL
        response = requests.get(url)

        # Check if the request was successful (status code 200) and raise an exception for bad responses
        response.raise_for_status()

        # Get the HTML content as a string
        html = response.text

    except requests.exceptions.RequestException as e:
        # Handle any errors that occurred during the request
        print(f"An error occurred: {e}")
    return html

In [166]:
url_base = 'https://www.dws.gov.za/iwqs/wms/data'
url_lp = f"{url_base}/WMS_pri_text_short.html"
landing_age_html = get_html(url_lp)

In [167]:
soup_lp = BeautifulSoup(landing_age_html, "lxml")
region_dicts = []

for index_tr, tr in enumerate(soup_lp.find_all("tr")):
    if index_tr <= 1:
        continue
    # print(tr)
    td_dict = {}
    for index_td, td in enumerate(tr.find_all("td")):

        if index_td <= 3:
            continue
        if index_td == 4:
            td_dict["region_desc"] = td.getText().split(": ")
            td_dict["region_letter"] = td_dict["region_desc"][0]
            td_dict["region_name"] = td_dict["region_desc"][1]

            td_dict["url_no_borehole"] = td.select_one("a")["href"]
            td_dict["url_no_borehole_full"] = f"{url_base}/{td_dict["url_no_borehole"]}"
        elif index_td == 5:
            td_dict["url_borehole"] = td.select_one("a")["href"]
            td_dict["url_borehole_full"] = f"{url_base}/{td_dict["url_borehole"]}"

    region_dicts.append(td_dict)
    # print(td_dict)
    # break

In [168]:
region_dicts

[{'region_desc': [' A', 'Limpopo'],
  'region_letter': ' A',
  'region_name': 'Limpopo',
  'url_no_borehole': 'A_reg_WMS_nobor.htm',
  'url_no_borehole_full': 'https://www.dws.gov.za/iwqs/wms/data/A_reg_WMS_nobor.htm',
  'url_borehole': 'A_reg_WMS_boreh.htm',
  'url_borehole_full': 'https://www.dws.gov.za/iwqs/wms/data/A_reg_WMS_boreh.htm'},
 {'region_desc': [' B', 'Olifants (E)'],
  'region_letter': ' B',
  'region_name': 'Olifants (E)',
  'url_no_borehole': 'B_reg_WMS_nobor.htm',
  'url_no_borehole_full': 'https://www.dws.gov.za/iwqs/wms/data/B_reg_WMS_nobor.htm',
  'url_borehole': 'B_reg_WMS_boreh.htm',
  'url_borehole_full': 'https://www.dws.gov.za/iwqs/wms/data/B_reg_WMS_boreh.htm'},
 {'region_desc': [' C', 'Vaal'],
  'region_letter': ' C',
  'region_name': 'Vaal',
  'url_no_borehole': 'C_reg_WMS_nobor.htm',
  'url_no_borehole_full': 'https://www.dws.gov.za/iwqs/wms/data/C_reg_WMS_nobor.htm',
  'url_borehole': 'C_reg_WMS_boreh.htm',
  'url_borehole_full': 'https://www.dws.gov.za/i

In [169]:
wq_data_start_date = date(2010, 1, 1)
wq_data_end_date = date(2016, 12, 31)

def extract_td_row_dict(tds):
    # print(f"\t{tds}")
    try:
        data_id = tds[0].getText().strip()
        data_zip_url = tds[2].select_one("a")["href"]
        data_desc = tds[3].getText().strip()
        data_type = tds[4].getText().strip()
        data_num_records = tds[5].getText().strip()
        data_first_date_str = tds[6].getText().strip()
        data_last_date_str = tds[7].getText().strip()
        data_med_ec = tds[8].getText().strip()
        data_flow = tds[9].getText().strip()
        data_lat = float(tds[10].getText().strip())
        data_lng = float(tds[11].getText().strip())

        format_string = "%Y-%m-%d"
        data_first_date = datetime.strptime(data_first_date_str, format_string).date()
        data_last_date = datetime.strptime(data_last_date_str, format_string).date()

        # Only keep records that fall within bounds of Water Qual data (2011 - 2014)
        if (wq_data_start_date <= data_first_date <= wq_data_end_date) or (wq_data_start_date <= data_last_date <= wq_data_end_date):
            td_row_dict = {
                "data_id": data_id,
                "data_zip_url": data_zip_url,
                "data_desc": data_desc,
                "data_type": data_type,
                "data_num_records": data_num_records,
                "data_first_date": data_first_date,
                "data_last_date": data_last_date,
                "data_med_ec": data_med_ec,
                "data_flow": data_flow,
                "data_lat": data_lat,
                "data_lng": data_lng,
            }
            return td_row_dict
        else:
            return None

    except Exception as e:
        # print(f"\tException Thrown, continuing ... \n\t\t{e}")
        return None

def extract_all_td_row_dicts(soup_obj):
    all_td_row_dicts = []
    for index_tr, tr in enumerate(soup_obj.find_all("tr")):
        if index_tr <= 1:
            continue

        tds = tr.find_all("td")
        td_row_dict = extract_td_row_dict(tds)
        if td_row_dict is not None:
            all_td_row_dicts.append(td_row_dict)

    return all_td_row_dicts

In [170]:
regions_row_dicts = {}

for region_dict in region_dicts:
    region_letter = region_dict["region_letter"]
    region_name = region_dict["region_name"]
    url_no_borehole_full = region_dict["url_no_borehole_full"]
    url_borehole_full = region_dict["url_borehole_full"]

    print(f"Processing {region_letter}: {region_name}...")
    os.makedirs(f"extracted_data/{region_name}", exist_ok=True)

    html_no_bore = get_html(url_no_borehole_full)
    html_bore = get_html(url_borehole_full)

    soup_no_bore = BeautifulSoup(html_no_bore, "lxml")
    soup_bore = BeautifulSoup(html_bore, "lxml")

    region_no_bore_row_dicts = extract_all_td_row_dicts(soup_no_bore)
    region_bore_row_dicts = extract_all_td_row_dicts(soup_bore)

    regions_row_dicts[region_name] = {
        "no_bore": region_no_bore_row_dicts,
        "bore": region_bore_row_dicts
    }

Processing  A: Limpopo...
Processing  B: Olifants (E)...
Processing  C: Vaal...
Processing  D: Orange...
Processing  E: Olifants (W) et al....
Processing  F: Buffels et al....
Processing  G: Great Berg et al....
Processing  H: Breë...
Processing  J: Gourits...
Processing  K: Kromme et al....
Processing  L: Gamtoos...
Processing  M: Swartkops...
Processing  N: Sundays...
Processing  O: Outside borders...
Processing  P: Bushmans et al....
Processing  Q: Great Fish...
Processing  R: Keiskamma et al....
Processing  S: Great Kei...
Processing  T: Mzimvubu et al....
Processing  U: uMngeni et al....
Processing  V: Thukela...
Processing  W: Phongolo et al....
Processing  X: Crocodile (E)...
Processing  Y: Limpopo +...
Processing  Z: Great Fish +...


In [171]:
regions_row_dicts["Limpopo"]

{'no_bore': [{'data_id': 'A21 100000752',
   'data_zip_url': 'https://www.dws.gov.za/iwqs/wms/data/A21/A21_100000752.zip',
   'data_desc': 'Hennops 01 on Skurweberg Dirt Road at Small Bridge',
   'data_type': 'Rivers',
   'data_num_records': '209',
   'data_first_date': datetime.date(2002, 6, 18),
   'data_last_date': datetime.date(2010, 2, 26),
   'data_med_ec': '67',
   'data_flow': 'A21',
   'data_lat': -25.83053,
   'data_lng': 28.11831},
  {'data_id': 'A21 100000766',
   'data_zip_url': 'https://www.dws.gov.za/iwqs/wms/data/A21/A21_100000766.zip',
   'data_desc': 'Rietvlei 06 downstream of Farm Dam on Road R23',
   'data_type': 'Rivers',
   'data_num_records': '186',
   'data_first_date': datetime.date(2002, 6, 18),
   'data_last_date': datetime.date(2010, 2, 26),
   'data_med_ec': '38',
   'data_flow': 'A21',
   'data_lat': -26.05225,
   'data_lng': 28.26608},
  {'data_id': 'A23 100000790',
   'data_zip_url': 'https://www.dws.gov.za/iwqs/wms/data/A23/A23_100000790.zip',
   'data_

In [172]:
def download_zip(url, save_path):
    """
    Downloads a zip file from a URL and saves it to a specified path.

    Args:
        url (str): The URL of the zip file.
        save_path (str): The local path to save the file (e.g., "downloaded_file.zip").
    """
    try:
        r = requests.get(url, stream=True)
        r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=128):
                f.write(chunk)
        # print(f"\tDownloaded '{save_path}' successfully.")
    except requests.exceptions.RequestException as e:
        print(f"\tError during download: {e}")

In [176]:
def extract_and_save_df(region_name, type, region_type_row_dicts):
    len_region_type_row_dicts = len(region_type_row_dicts)
    print(f"\tProcessing {region_name} - {type} ({len_region_type_row_dicts} records to process)...")
    full_df = None
    for index, region_type_row_dict in enumerate(region_type_row_dicts):
        # print(f"\t{region_dict_no_bore}")
        processing_dir = "processing"
        tmp_zip_name = f"{processing_dir}/processing.zip"
        data_id = region_type_row_dict["data_id"]
        data_zip_url = region_type_row_dict["data_zip_url"]
        data_desc = region_type_row_dict["data_desc"]
        data_num_records = region_type_row_dict["data_num_records"]
        data_first_date = region_type_row_dict["data_first_date"]
        data_last_date = region_type_row_dict["data_last_date"]
        data_med_ec = region_type_row_dict["data_med_ec"]
        data_flow = region_type_row_dict["data_flow"]
        data_lat = region_type_row_dict["data_lat"]
        data_lng = region_type_row_dict["data_lng"]

        try:
            print(f"\t\tProcessing {region_name} - {data_id} ({index}/{len_region_type_row_dicts})... ", end="")
            download_zip(region_type_row_dict["data_zip_url"], save_path=tmp_zip_name)
            with zipfile.ZipFile(tmp_zip_name, 'r') as zip_ref:
                zip_ref.extractall(processing_dir)
        except Exception as e:
            print(f"\t\t\t Exception (will skip) : {e} ")
            continue

        for filename in os.listdir(processing_dir):
            if filename.endswith(".csv"):
                tmp_df = pd.read_csv(f"{processing_dir}/{filename}")
                tmp_df["data_id"] = data_id
                tmp_df["data_zip_url"] = data_zip_url
                tmp_df["data_desc"] = data_desc
                tmp_df["data_num_records"] = data_num_records
                tmp_df["data_first_date"] = data_first_date
                tmp_df["data_last_date"] = data_last_date
                tmp_df["data_med_ec"] = data_med_ec
                tmp_df["data_flow"] = data_flow
                tmp_df["data_lat"] = data_lat
                tmp_df["data_lng"] = data_lng

                if full_df is None:
                    full_df = tmp_df
                else:
                    full_df = pd.concat([full_df, tmp_df], axis=0, ignore_index=True)

        for filename in os.listdir(processing_dir):
            os.remove(f"{processing_dir}/{filename}")

        full_df["region_name"] = region_name
        full_df["bore_type"] = type

        full_df.to_csv(f"extracted_data/{region_name}/{type}.csv", index=False)
        print("Done")


In [177]:
for region_name in regions_row_dicts.keys():
    print(f"Processing {region_name}...")
    region_row_dicts = regions_row_dicts[region_name]
    region_row_dicts_no_bore = region_row_dicts["no_bore"]
    region_row_dicts_bore = region_row_dicts["bore"]

    extract_and_save_df(region_name, "no_bore", region_row_dicts_no_bore)
    extract_and_save_df(region_name, "bore", region_row_dicts_bore)

Processing Limpopo...
	Processing Limpopo - no_bore (200 records to process)...
		Processing Limpopo - A21 100000752 (0/200)... Done
		Processing Limpopo - A21 100000766 (1/200)... Done
		Processing Limpopo - A23 100000790 (2/200)... Done
		Processing Limpopo - A23 100000881 (3/200)... Done
		Processing Limpopo - A23 100000882 (4/200)... Done
		Processing Limpopo - A23 100000883 (5/200)... Done
		Processing Limpopo - A23 100000884 (6/200)... Done
		Processing Limpopo - A23 100000885 (7/200)... Done
		Processing Limpopo - A23 100000886 (8/200)... Done
		Processing Limpopo - A23 100000887 (9/200)... Done
		Processing Limpopo - A23 100000888 (10/200)... Done
		Processing Limpopo - A23 100000967 (11/200)... Done
		Processing Limpopo - A21 189445 (12/200)... Done
		Processing Limpopo - A22 189036 (13/200)... Done
		Processing Limpopo - A22 189442 (14/200)... Done
		Processing Limpopo - A22 189032 (15/200)... Done
		Processing Limpopo - A22 188072 (16/200)... Done
		Processing Limpopo - A22 